# NHANES risk-risk correlation matrix — overview

Builds the full pairwise correlation matrix that the consuming simulation project's `RiskCorrelation` component consumes at simulant initialization. Until now most of the off-diagonals were literature stand-ins; this analysis replaces them with NHANES-derived weighted Spearman estimates wherever the data permit, and documents which pairs are (and must remain) literature-anchored.

## Risks in the correlation matrix

Eight risks: BMI, LDL-C, SBP, FPG, smoking, Lp(a), liver stiffness, kidney_dysfunction. Of these, **smoking and kidney_dysfunction are categorical**; we estimate their propensity-level correlations against the continuous risks via rank correlation on the underlying continuous proxy (current=1/former=2/never=3 ordinal for smoking; eGFR for kidney_dysfunction).

## The data-availability gotcha

No single NHANES cycle carries every risk:

| Cycle | BMI | LDL-C | SBP | FPG | Smoking | Creatinine | Lp(a) | LSM |
| :--- | :--: | :--: | :--: | :--: | :--: | :--: | :--: | :--: |
| Continuous 1999–2018 | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | — | — |
| NHANES III Phase II (1991–94) | ✓ | ✓ | ✓ | ✓ | ✓ | (yes, but unused) | **✓** | — |
| 2017 – March 2020 (P_LUX) | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | — | **✓** |

So the matrix is assembled from three sources, with one structurally-missing cell: **Lp(a) ↔ LSM is never measured in the same respondent.** That pair stays a literature/assumption value (the project currently uses 0.0, defensible because Lp(a) is largely genetic and LSM reflects metabolic/fibrosis pathways).

## Notebooks

1. `01_continuous_block.ipynb` — 10 pairs among {BMI, LDL-C, SBP, FPG, smoking} + 5 pairs against kidney_dysfunction-via-eGFR, all from continuous NHANES (pooled 2007–2018; creatinine via CKD-EPI 2021 for the kidney channel).
2. `02_lpa_pairs.ipynb` — Lp(a) ↔ {BMI, LDL-C, SBP, FPG, smoking, eGFR} from NHANES III Phase II. 6 pairs.
3. `03_lsm_pairs.ipynb` — LSM ↔ {BMI, LDL-C, SBP, FPG, smoking, eGFR} from the 2017–March 2020 P_LUX merge. 6 pairs.
4. `04_assemble_matrix.ipynb` — combines into the 8 × 8 final matrix; flags Lp(a) ↔ LSM as literature 0.0; emits `outputs/correlation_matrix.csv` for the project loader.

## Stratification: is one matrix per simulation defensible?

The simulation reads one correlation matrix at initialization and applies it uniformly. Two checks per pair, where data permit:

- **Age stratification.** Compute the trial-band 65–80 estimate (the project's primary value), plus a younger-adult slice (40–64) and a balance-of-adults slice. If the three estimates agree within sampling noise, a single matrix is fine; if they diverge by more than ~0.10 in ρ, the project needs an age-stratified matrix or to accept the trial-band value as a localized estimate.
- **Time-trend (per-cycle).** For the continuous block, compute ρ separately by 6-year bin (2007–2012, 2013–2018) — the trial period is ~2025–2030, so the most recent NHANES cycles are the right anchor. If recent cycles drift away from earlier ones, anchor on the late slice. (Lp(a) and LSM each come from a single source cycle, so time-trend isn't checkable for those pairs; we note this and move on.)

## Methods

- **Weighted Spearman.** Rank-based, robust to skewness — important because Lp(a) and LSM are heavy-tailed and metabolic-cluster pairs are confounded by treatment at the upper tail.
- **Survey weights.** MEC examination weight (`WTMEC2YR` for continuous cycles, divided by the number of cycles pooled per the NCHS analytic guidelines; `WTPFEX6` for NHANES III Phase II; `WTMECPRP` for the pre-pandemic combined release).
- **Variance.** Paired-PSU jackknife on the masked (`SDMVSTRA` × `SDMVPSU` for continuous, `SDPSTRA6` × `SDPPSU6` for NHANES III) variance design. Per-pair SE → 95 % CI for the headline estimate; for stratification checks we report point estimates only (CIs would clutter the table).
- **Trial-band primary slice.** Age 65–80, both sexes pooled.

## Caveats

- **Treatment confounding.** Older adults often take statins, antihypertensives, and metformin, which compress the upper tail of LDL-C / SBP / FPG and can flip pairwise correlations from their underlying biological values. The matrix reports the **as-observed** correlation, matching what the simulation will see if it initializes from GBD risk distributions (which already bake in current treatment). `nhanes_bmi_ldlc_correlation/02_treatment_deletion.ipynb` did a sensitivity analysis for the BMI ↔ LDL-C pair; the same logic generalizes but is not repeated here.
- **Sign convention for categorical risks.** VPH's polytomous PPF maps **low propensity → cat1 (worst)**. To translate value-level signs (e.g. "current smokers tend to have higher BMI") into propensity-level signs, the categorical-side variable is signed so that lower-rank = worst-stage. Documented per pair in the assembly notebook.
- **Static matrix.** The simulation reads one matrix at simulant initialization. Correlations may drift over the trial's 5-year follow-up (treatment uptake, age progression), but the project does not currently update the joint distribution mid-sim.